In [1]:
import os
import pandas as pd
from datetime import datetime, UTC
from truthbrush.api import Api
import time

In [2]:
os.environ['TRUTHSOCIAL_USERNAME'] = 'epurevsuren'
os.environ['TRUTHSOCIAL_PASSWORD'] = '7Qy#J9d3Jmw7!Tx'

In [3]:
# -------- CONFIG --------
USERNAME = "realDonaldTrump"
CREATED_AFTER = datetime(
    2026, 5, 13,
    tzinfo=UTC
)

CREATED_BEFORE = datetime(
    2026, 5, 23,
    tzinfo=UTC
)
OUTPUT_FILE = "trump_truths.csv"

In [4]:
# -------- INIT API --------
api = Api(username=os.environ['TRUTHSOCIAL_USERNAME'], password=os.environ['TRUTHSOCIAL_PASSWORD'])
results = api.pull_statuses("@realDonaldTrump")

def safe_get(d, key, default=None):
    return d.get(key, default)

In [5]:
import os
import time
import random
import re

from bs4 import BeautifulSoup
from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings(
    "ignore",
    category=MarkupResemblesLocatorWarning
)

OUTPUT_FILE = "trump_truths.csv"


def normalize_text(html):

    if not html:
        return ""

    soup = BeautifulSoup(html, "html.parser")

    text = soup.get_text()

    text = text.replace("\r\n", " ")
    text = text.replace("\r", " ")

    text = text.replace("\u2028", " ")
    text = text.replace("\u2029", " ")

    text = text.replace("\u00A0", " ")

    text = re.sub(r"https://\s+", "https://", text)
    text = re.sub(r"http://\s+", "http://", text)

    text = re.sub(r"([^\s])https://", r"\1 https://", text)
    text = re.sub(r"([^\s])http://", r"\1 http://", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text)
    
    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\n\s*\n", " ", text)

    text = re.sub(r"\n{3,}", " ", text)

    return text.strip()


def load_existing_ids():

    if not os.path.exists(OUTPUT_FILE):
        return set()

    df = pd.read_csv(OUTPUT_FILE)

    return set(df["id"].astype(str))


def save_checkpoint(rows):

    if not rows:
        return

    df = pd.DataFrame(rows)

    file_exists = os.path.exists(OUTPUT_FILE)

    df.to_csv(
        OUTPUT_FILE,
        mode="a",
        header=not file_exists,
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n"
    )


def fetch_posts():

    rows = []

    processed = 0
    saved = 0

    results = api.pull_statuses(
        USERNAME,
        created_after=CREATED_AFTER,
        created_before=CREATED_BEFORE
    )

    for post in results:

        try:

            # Skip replies
            if post.get("in_reply_to_id") is not None:
                continue

            # Skip retruths
            if post.get("reblog") is not None:
                continue

            cleaned_text = normalize_text(
                post.get("content", "")
            )

            if not cleaned_text:
                continue

            rows.append({
                "id": str(post.get("id")),
                "date": post.get("created_at"),
                "text": cleaned_text,
                "url": post.get("url"),
                "favorites": post.get("favourites_count"),
                "retweets": post.get("reblogs_count"),
                "replies": post.get("replies_count")
            })

            processed += 1
            saved += 1

            # Save every 100 posts
            if saved >= 100:

                print(f"Saving checkpoint at {processed} posts")

                save_checkpoint(rows)

                rows = []
                saved = 0

                sleep_time = random.uniform(1.5, 3.5)

                print(f"Sleeping {sleep_time:.1f} seconds")

                time.sleep(sleep_time)

        except Exception as e:

            print("Error processing post:")
            print(e)

            # Save before crash
            save_checkpoint(rows)

            raise

    # Final save
    save_checkpoint(rows)

    print(f"Finished scraping {processed} posts")

In [6]:
def main():
    rows = fetch_posts()


In [7]:
if __name__ == "__main__":
    main()

2026-05-23 19:09:49.662 | WARNING  | truthbrush.api:__check_login:103 - Using token tEYu3EiuUcWC5gsy-BYYAqfQR69JY5ZTbOf3JETVJWA


Saving checkpoint at 100 posts
Sleeping 2.4 seconds
Finished scraping 125 posts
